# Chạy pipeline trên Kaggle
Giải nén project vào `/kaggle/working/Helmet_Django` (hoặc sửa PROJECT_DIR bên dưới). Cung cấp hai pipeline model thật. Đây là inference ảnh toàn cảnh, không phải train.

Widget upload phụ thuộc frontend; dùng đường dẫn ảnh luôn được. Camera snapshot dùng app.py trên máy local, không dùng webcam trực tiếp qua notebook Kaggle.

In [ ]:
from pathlib import Path
import sys
PROJECT_DIR = Path('/kaggle/working/Helmet_Django')
if not (PROJECT_DIR / 'src/pipeline.py').is_file():
    raise FileNotFoundError('Giải nén project và sửa PROJECT_DIR cho đúng.')
sys.path.insert(0, str(PROJECT_DIR))
from src.pipeline import HelmetPipeline
from src.head_detector import DetectionConfig
from PIL import Image, ImageOps
import numpy as np
from IPython.display import display
import json

HEAD_MODEL = PROJECT_DIR / 'models/head_svm.joblib'
HELMET_MODEL = PROJECT_DIR / 'models/helmet_svm.joblib'
engine = HelmetPipeline(HEAD_MODEL, HELMET_MODEL)
config = DetectionConfig(min_side=32, max_side=256, score_threshold=0.0)
OUTPUT = Path('/kaggle/working/detection_results')
OUTPUT.mkdir(parents=True, exist_ok=True)


In [ ]:
def run_image(source):
    with Image.open(source) as image:
        image = ImageOps.exif_transpose(image)
        if image.mode == 'RGBA' or 'transparency' in image.info:
            rgba = image.convert('RGBA')
            image = Image.alpha_composite(Image.new('RGBA', rgba.size, (255,255,255,255)), rgba)
        rgb = np.asarray(image.convert('RGB'))
    result = engine.run(rgb, config, padding=0.0)
    display(Image.fromarray(result['annotated_rgb']))
    print(result['stats'])
    print(result['detections'])
    Image.fromarray(result['annotated_rgb']).save(OUTPUT / 'result.png')
    (OUTPUT / 'detections.json').write_text(json.dumps({'detections': result['detections'], 'stats': result['stats']}, indent=2))
    # Each call replaces result.png/json; crop names are not reused across runs.
    import uuid
    crop_dir = OUTPUT / ('crops_' + uuid.uuid4().hex[:8])
    crop_dir.mkdir()
    for row, crop in zip(result['detections'], result['crops_rgb']):
        Image.fromarray(crop).save(crop_dir / f"{row['id']:03d}_{row['label']}.png")
    return result


## Cách 1: dùng đường dẫn ảnh đầy đủ (sửa IMAGE_PATH rồi chạy)

In [ ]:
IMAGE_PATH = Path('/kaggle/input/your-dataset/example.jpg')
if IMAGE_PATH.is_file():
    result = run_image(IMAGE_PATH)
else:
    print('Hãy sửa IMAGE_PATH thành đường dẫn ảnh toàn cảnh của bạn.')


## Cách 2: upload ảnh qua widget (tùy frontend Kaggle)

In [ ]:
import ipywidgets as widgets
from io import BytesIO
uploader = widgets.FileUpload(accept='image/*', multiple=False)
button = widgets.Button(description='Phát hiện ảnh upload', button_style='primary')
out = widgets.Output()
def on_click(_):
    with out:
        out.clear_output()
        value = uploader.value
        if not value:
            print('Chọn một ảnh trước.'); return
        item = next(iter(value.values())) if isinstance(value, dict) else value[0]
        try:
            run_image(BytesIO(bytes(item['content'])))
        except Exception as exc:
            print(f'Lỗi: {exc}')
button.on_click(on_click)
display(uploader, button, out)
